In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio langchain langchain-huggingface

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
import asyncio, json, re, logging, time
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from transformers import pipeline

# 1. Configurar pipeline básico de generación de texto
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# 2. Envolver con LangChain pasando max_new_tokens a través de pipeline_kwargs
hf_llm = HuggingFacePipeline(pipeline=pipe, pipeline_kwargs={"max_new_tokens": 1024})
chat_model = ChatHuggingFace(llm=hf_llm)

class LangChainAgent:
    def __init__(self, model):
        self.model = model

    def complete(self, messages):
        lc_messages = []
        for msg in messages:
            if msg.role == 'system':
                lc_messages.append(SystemMessage(content=msg.content))
            elif msg.role == 'user':
                lc_messages.append(HumanMessage(content=msg.content))
            elif msg.role == 'assistant':
                lc_messages.append(AIMessage(content=msg.content))
        
        response_msg = self.model.invoke(lc_messages)
        response = response_msg.content
        return re.sub(r'<\|im_end\|>', '', response)


In [4]:
llm = LangChainAgent(chat_model)

from pydantic import BaseModel
from typing import List, Optional, Literal
from datetime import datetime

class Message(BaseModel):
    role: Literal['system', 'user']  # roles comunes en chats
    content: str

class ChatRequest(BaseModel):
    messages: List[Message] = []  # Valor por defecto lista vacía



In [5]:
from fastapi import FastAPI, Request
from pyngrok import ngrok
import nest_asyncio
import uvicorn

ngrok.set_auth_token("344HT0PzWr1pGVLwZBa7KWXfxXE_4FMsfMKfHFpG8ZAQXrpS7")


nest_asyncio.apply()  # Permite correr uvicorn en el loop de Colab

app = FastAPI()


@app.get("/")
def home():
    return {"message": "Hola desde Colab + FastAPI!"}



@app.post("/chat")
async def chat(body: ChatRequest):

    response = llm.complete(body.messages)
    print(response)

    return {
        "response": response
    }

In [ ]:
from pyngrok import ngrok

# Cerrar cualquier túnel previo colgado para evitar el error ERR_NGROK_334
try:
    print("Cerrando túneles activos anteriores...")
    ngrok.kill()
except Exception as e:
    print("No se encontraron túneles previos activos.")

# Crear túnel en el puerto 8000
public_url = ngrok.connect(8000)
print("URL pública:", public_url)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()


Cerrando túneles activos anteriores...


INFO:     Started server process [215]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


URL pública: NgrokTunnel: "https://transmarginally-unrebuffed-else.ngrok-free.dev" -> "http://localhost:8000"
INFO:     2803:9800:b882:7de0:736:244a:527c:24ff:0 - "GET / HTTP/1.1" 200 OK
INFO:     2803:9800:b882:7de0:736:244a:527c:24ff:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
Eres un evaluador semántico especializado en respuestas académicas.
Recibirás una PREGUNTA, una RESPUESTA de un estudiante y un CONCEPTO a evaluar.
Tu tarea consiste exclusivamente en determinar si la respuesta expresa o no dicho concepto.

## OBJETIVO
No debes calificar la respuesta.
No debes explicar tu razonamiento.
No debes agregar texto nuevo.
Debes responder únicamente con "sí" o "no".

## REGLA PRINCIPAL
Solamente debes evaluar el concepto recibido.
Está prohibido:
- Evaluar conceptos no proporcionados
- Inferir categorías no definidas
- Responder con texto adicional

## CRITERIOS DE DETECCIÓN
La coincidencia es semántica, no textual.
Considera un concepto presente cuando:
- La idea principal coincide con el concepto
- El significado es equivalente
- Puede estar parafraseado
- Puede utilizar sinónimos
- Puede tener errores gramaticales menores

No marques un concepto como presente cuando:
- Aparece solamente una palabra clave sin desarrollar la idea
- La expli

...

<|im_start|>system
Eres un evaluador semántico especializado en respuestas académicas.
Recibirás una PREGUNTA, una RESPUESTA de un estudiante y un CONCEPTO a evaluar.
Tu tarea consiste exclusivamente en determinar si la respuesta expresa o no dicho concepto.

## OBJETIVO
No debes calificar la respuesta.
No debes explicar tu razonamiento.
No debes agregar texto nuevo.
Debes responder únicamente con "sí" o "no".

## REGLA PRINCIPAL
Solamente debes evaluar el concepto recibido.
Está prohibido:
- Evaluar conceptos no proporcionados
- Inferir categorías no definidas
- Responder con texto adicional

## CRITERIOS DE DETECCIÓN
La coincidencia es semántica, no textual.
Considera un concepto presente cuando:
- La idea principal coincide con el concepto
- El significado es equivalente
- Puede estar parafraseado
- Puede utilizar sinónimos
- Puede tener errores gramaticales menores

No marques un concepto como presente cuando:
- Aparece solamente una palabra clave sin desarrollar la idea
- La expli

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
Eres un evaluador semántico especializado en respuestas académicas.
Recibirás una PREGUNTA, una RESPUESTA de un estudiante y un CONCEPTO a evaluar.
Tu tarea consiste exclusivamente en determinar si la respuesta expresa o no dicho concepto.

## OBJETIVO
No debes calificar la respuesta.
No debes explicar tu razonamiento.
No debes agregar texto nuevo.
Debes responder únicamente con "sí" o "no".

## REGLA PRINCIPAL
Solamente debes evaluar el concepto recibido.
Está prohibido:
- Evaluar conceptos no proporcionados
- Inferir categorías no definidas
- Responder con texto adicional

## CRITERIOS DE DETECCIÓN
La coincidencia es semántica, no textual.
Considera un concepto presente cuando:
- La idea principal coincide con el concepto
- El significado es equivalente
- Puede estar parafraseado
- Puede utilizar sinónimos
- Puede tener errores gramaticales menores

No marques un concepto como presente cuando:
- Aparece solamente una palabra clave sin desarrollar la idea
- La expli

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
Eres un Agente Evaluador de la UTN. Analiza de forma minuciosa.
<|im_start|>user
Pregunta: ¿Qué es el ciclo TDD?
Respuesta: TDD consiste en escribir los test primero en rojo, luego implementar el código para que pase a verde y finalmente refactorizar.

Instrucciones de formato:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"razonamiento_previo": {"description": "Chain-of-thought", "title": "Razonamiento Previo", "type": "string"}, "conceptos_clave_encontrados": {"description": "Conceptos encontrados", "items": {"type": "string"}, "title": "Conceptos Clav